In [1]:
import pandas as pd

# Load raw data
df = pd.read_csv("iris.data", header=None)

# Assign column names
df.columns = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "class"
]

# Drop class label
data = df.drop(columns=["class"])

# Save as CSV (optional but recommended)
data.to_csv("iris.csv", index=False)

print(data.head())

   sepal_length  sepal_width  petal_length  petal_width
0           5.1          3.5           1.4          0.2
1           4.9          3.0           1.4          0.2
2           4.7          3.2           1.3          0.2
3           4.6          3.1           1.5          0.2
4           5.0          3.6           1.4          0.2


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components

In [3]:
# Load iris.csv (features only)
df = pd.read_csv("iris.csv")

print("First 5 rows:")
display(df.head())

# Convert to numpy
data = df.values

First 5 rows:


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [4]:
# Load iris.csv (features only)
df = pd.read_csv("iris.csv")

print("First 5 rows:")
display(df.head())

# Convert to numpy
data = df.values

First 5 rows:


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [7]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

print("Data normalized successfully!")

Data normalized successfully!


In [8]:
dist_matrix = pairwise_distances(data_scaled, metric='euclidean')

print("Distance matrix shape:", dist_matrix.shape)

Distance matrix shape: (150, 150)


In [9]:
mst_sparse = minimum_spanning_tree(dist_matrix)
mst = mst_sparse.toarray()

print("MST constructed!")

MST constructed!


In [10]:
edges = np.array(np.nonzero(mst)).T

edge_list = [(u, v, mst[u, v]) for u, v in edges]

# Sort edges by weight (descending)
edge_list = sorted(edge_list, key=lambda x: -x[2])

print("Top 5 largest edges:")
print(edge_list[:5])

Top 5 largest edges:
[(np.int64(41), np.int64(57), np.float64(1.558495763239299)), (np.int64(8), np.int64(41), np.float64(1.4010369248354528)), (np.int64(109), np.int64(144), np.float64(0.9490556589507729)), (np.int64(109), np.int64(117), np.float64(0.9236939021354718)), (np.int64(59), np.int64(106), np.float64(0.786323075838847))]


In [11]:
k = 3  # number of clusters

mst_cut = mst.copy()

for i in range(k - 1):
    u, v, w = edge_list[i]
    mst_cut[u, v] = 0

print(f"Removed {k-1} largest edges")

Removed 2 largest edges


In [12]:
n_components, mst_labels = connected_components(mst_cut)

print("Number of clusters (MST):", n_components)
print("Cluster labels:", np.unique(mst_labels))

Number of clusters (MST): 3
Cluster labels: [0 1 2]


In [13]:
mst_df = pd.DataFrame({
    "SampleId": range(len(mst_labels)),
    "ClusterLabel": mst_labels
})

mst_df.to_csv("mst_clusters.csv", index=False)

print("MST clusters saved!")

MST clusters saved!


In [14]:
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(data_scaled)

print("K-Means clustering done!")

K-Means clustering done!


In [15]:
kmeans_df = pd.DataFrame({
    "SampleId": range(len(kmeans_labels)),
    "ClusterLabel": kmeans_labels
})

kmeans_df.to_csv("kmeans_clusters.csv", index=False)

print("K-Means clusters saved!")

K-Means clusters saved!


In [16]:
mst_score = silhouette_score(data_scaled, mst_labels)
kmeans_score = silhouette_score(data_scaled, kmeans_labels)

print("MST Silhouette Score     :", round(mst_score, 4))
print("K-Means Silhouette Score :", round(kmeans_score, 4))

MST Silhouette Score     : 0.5029
K-Means Silhouette Score : 0.459


In [17]:
print("""
MST clustering can capture non-spherical clusters because it connects points based on
minimum distances without assuming any cluster shape. By removing the longest edges,
it separates weakly connected regions, allowing detection of irregular structures.

However, MST is sensitive to noise and chaining effects, where clusters may be incorrectly
connected by a few intermediate points.

K-Means performs better on datasets like Iris where clusters are compact and spherical,
but it fails on non-convex shapes.
""")


MST clustering can capture non-spherical clusters because it connects points based on
minimum distances without assuming any cluster shape. By removing the longest edges,
it separates weakly connected regions, allowing detection of irregular structures.

However, MST is sensitive to noise and chaining effects, where clusters may be incorrectly
connected by a few intermediate points.

K-Means performs better on datasets like Iris where clusters are compact and spherical,
but it fails on non-convex shapes.

